<a href="https://colab.research.google.com/github/jonathan-ha1/Problem-Set-5/blob/main/Econ_3916_PS5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import metrics

RANDOM_STATE = 42
rng = np.random.default_rng(seed=RANDOM_STATE)
n = 6000

income = rng.lognormal(10.8, 0.45, n)
dti = rng.beta(2, 5, n) * 0.7
history = rng.lognormal(2.3, 0.5, n)
prior = rng.poisson(0.3, n)
loan_amt = rng.lognormal(9.5, 0.6, n)
noise_1 = rng.normal(0, 1, n)
noise_2 = rng.beta(2, 5, n)
noise_3 = rng.poisson(2, n)

logit = (-3.8 - 0.9 * (np.log(income) - 10.8) + 4.0 * dti
         - 0.05 * (history - 10) + 0.6 * prior
         + 0.4 * (np.log(loan_amt) - 9.5))
prob = 1 / (1 + np.exp(-logit))
default = rng.binomial(1, prob)

loans = pd.DataFrame({"log_income": np.log(income), "dti": dti, "history": history,
                      "prior": prior, "log_loan": np.log(loan_amt),
                      "noise_1": noise_1, "noise_2": noise_2, "noise_3": noise_3,
                      "default": default})
FEATURES = ["log_income", "dti", "history", "prior", "log_loan",
            "noise_1", "noise_2", "noise_3"]
print(f"n = {n:,}, default rate = {default.mean():.1%}")

X_tr, X_te, y_tr, y_te = train_test_split(
    loans[FEATURES], loans["default"],
    test_size=0.3, random_state=RANDOM_STATE, stratify=loans["default"])
print(f"train = {len(X_tr):,}, test = {len(X_te):,}")

n = 6,000, default rate = 7.0%
train = 4,200, test = 1,800


I simulated 6,000 loan applications where about 7% of people default, and I based default on income, debt-to-income, credit history length, past delinquencies and loan amount, plus three random noise features that have no effect on default. Before looking at anything, I split the data into 4,200 applications for training and 1,800 for testing, keeping the default rate the same in both, so every number I report after this comes from the test set.


In [2]:
base_acc = 1 - y_te.mean()
null_pred = np.zeros(len(y_te))

print(f"Null model test accuracy: {metrics.accuracy_score(y_te, null_pred):.1%}")
print(f"Defaults caught (recall): {metrics.recall_score(y_te, null_pred):.1%}")
print(metrics.confusion_matrix(y_te, null_pred))

Null model test accuracy: 92.9%
Defaults caught (recall): 0.0%
[[1673    0]
 [ 127    0]]


The null model predicts no default for everyone and gets 92.9% accuracy on the test set, which is almost the same as the vendor's 93% claim. It catches none of the 127 real defaults, so its recall is 0%, which means the 93% figure alone says nothing about whether the vendor's model learned anything, and any real model has to clearly beat this floor to prove it's worth using.


In [3]:
#Phase 2
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

logreg = LogisticRegression(penalty=None, max_iter=1000)
logreg.fit(X_tr_s, y_tr)
logreg_prob = logreg.predict_proba(X_te_s)[:, 1]
logreg_pred = logreg.predict(X_te_s)

print(f"Logistic test accuracy: {metrics.accuracy_score(y_te, logreg_pred):.1%}")
print(f"Logistic test AUC: {metrics.roc_auc_score(y_te, logreg_prob):.3f}")
print(f"Defaults caught (recall): {metrics.recall_score(y_te, logreg_pred):.1%}")

Logistic test accuracy: 92.9%
Logistic test AUC: 0.713
Defaults caught (recall): 0.8%


The plain logistic regression gets 92.9% accuracy on the test set, the same as the null model, and at the usual 0.5 cutoff it only catches one of the 127 defaults, so judging by accuracy alone it would look like it learned nothing. Its AUC of 0.713 shows otherwise, since the model is ranking risky applicants above safe ones, and that's exactly the kind of signal the vendor's accuracy number hides.

In [4]:
from sklearn.linear_model import LogisticRegressionCV

lasso = LogisticRegressionCV(Cs=np.logspace(-3, 2, 30), cv=5, penalty="l1", solver="liblinear", scoring="roc_auc", random_state=RANDOM_STATE)
lasso.fit(X_tr_s, y_tr)
lasso_prob = lasso.predict_proba(X_te_s)[:, 1]
lasso_coefs = pd.Series(lasso.coef_[0], index=FEATURES)

print(f"Best C: {lasso.C_[0]:.4f}")
print(f"LASSO test AUC: {metrics.roc_auc_score(y_te, lasso_prob):.3f}")
print(lasso_coefs)
print(f"Driven to zero: {list(lasso_coefs[lasso_coefs == 0].index)}")

Best C: 0.1743
LASSO test AUC: 0.714
log_income   -0.301240
dti           0.440677
history      -0.215191
prior         0.273035
log_loan      0.202022
noise_1       0.000000
noise_2      -0.013911
noise_3      -0.028879
dtype: float64
Driven to zero: ['noise_1']


LASSO dropped only noise_1, so it caught 1 of the 3 fake features and kept all 5 real ones. It shrank noise_2 and noise_3 close to zero but didn't remove them, because the penalty that gave the best AUC was too light to cut them completely, and its test AUC of 0.714 is about the same as the plain logistic model.

In [5]:
ridge = LogisticRegressionCV(Cs=np.logspace(-3, 2, 30), cv=5, penalty="l2", solver="liblinear", scoring="roc_auc", random_state=RANDOM_STATE)
ridge.fit(X_tr_s, y_tr)
ridge_prob = ridge.predict_proba(X_te_s)[:, 1]
ridge_coefs = pd.Series(ridge.coef_[0], index=FEATURES)

print(f"Best C: {ridge.C_[0]:.4f}")
print(f"Ridge test AUC: {metrics.roc_auc_score(y_te, ridge_prob):.3f}")
print(ridge_coefs)
print(f"Driven to zero: {list(ridge_coefs[ridge_coefs == 0].index)}")

Best C: 30.3920
Ridge test AUC: 0.713
log_income   -0.331022
dti           0.471751
history      -0.251918
prior         0.296299
log_loan      0.229485
noise_1       0.016444
noise_2      -0.037114
noise_3      -0.053667
dtype: float64
Driven to zero: []


Ridge kept all 8 features without zeroing any of them, and since cross-validation picked a very light penalty, its test AUC of 0.713 matches the plain logistic model. Ridge shrinks coefficients but never zeros them because its penalty gets weaker as a coefficient gets smaller, so it never pushes hard enough to reach zero, while LASSO pushes just as hard on small coefficients as big ones, which lets it cut them out completely. LASSO makes more sense when a lot of features are probably useless and the lender wants a simple model that's easy to explain to a regulator, while Ridge makes more sense when most features matter or are related to each other, since it keeps all of them instead of dropping one.

In [6]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=500, oob_score=True, bootstrap=True, random_state=RANDOM_STATE)
rf.fit(X_tr, y_tr)
rf_prob = rf.predict_proba(X_te)[:, 1]
rf_pred = rf.predict(X_te)

print(f"OOB accuracy: {rf.oob_score_:.1%}")
print(f"Forest test accuracy: {metrics.accuracy_score(y_te, rf_pred):.1%}")
print(f"Forest test AUC: {metrics.roc_auc_score(y_te, rf_prob):.3f}")
print(f"LASSO test AUC: {metrics.roc_auc_score(y_te, lasso_prob):.3f}")

OOB accuracy: 92.9%
Forest test accuracy: 92.9%
Forest test AUC: 0.676
LASSO test AUC: 0.714


The forest's OOB accuracy and test accuracy were both 92.9%, and they should match because OOB checks each row using only the trees that never saw it, which makes it like a free test, so a big gap between them would mean the model memorized the training data or the test data looks different. The forest's test AUC of 0.676 is lower than LASSO's 0.714, so the bigger, slower model does worse than the simple one, which works out well for the lender since the simple model is also easier to explain to a regulator.

In [7]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_te, y_te, n_repeats=10, scoring="roc_auc", random_state=RANDOM_STATE)
importances = pd.DataFrame({"MDI": rf.feature_importances_, "Permutation": perm.importances_mean}, index=FEATURES)
print(importances.sort_values("MDI", ascending=False))


                 MDI  Permutation
dti         0.173312     0.073321
log_income  0.155965     0.044360
log_loan    0.153575     0.013002
history     0.151950    -0.005273
noise_1     0.141172    -0.003579
noise_2     0.137202     0.007755
noise_3     0.054543    -0.005568
prior       0.032282     0.019783


The two methods don't agree, since MDI ranks noise_1 and noise_2 almost as high as the real features, but permutation shows they barely matter for the test AUC. Prior delinquencies is the opposite, ranking last on MDI but near the top on permutation, because MDI gives extra credit to features with lots of different values, and prior only has a few whole numbers.

Just because a feature helps predict default doesn't mean the lender can change it, since turning someone down won't raise their income. Loan amount is different because the lender picks it, so instead of turning down a risky applicant, the lender could offer a smaller loan, but the model only shows that bigger loans go with more defaults, not that a smaller loan would make that person safer.


In [8]:
#Phase 3
lasso_pred = lasso.predict(X_te_s)

print(metrics.confusion_matrix(y_te, lasso_pred))
print(f"Precision: {metrics.precision_score(y_te, lasso_pred, zero_division=0):.3f}")
print(f"Recall: {metrics.recall_score(y_te, lasso_pred):.3f}")
print(f"F1: {metrics.f1_score(y_te, lasso_pred):.3f}")
print(f"ROC AUC: {metrics.roc_auc_score(y_te, lasso_prob):.3f}")
print(f"PR AUC: {metrics.average_precision_score(y_te, lasso_prob):.3f}")

[[1673    0]
 [ 126    1]]
Precision: 1.000
Recall: 0.008
F1: 0.016
ROC AUC: 0.714
PR AUC: 0.186


At the 0.5 cutoff, LASSO only flags one applicant and that person did default, so precision is 1.000 but recall is 0.008 and F1 is 0.016, meaning it misses 126 of the 127 defaults. The ROC AUC of 0.714 looks decent because it compares false alarms to the 1,673 people who didn't default, so even a lot of false alarms barely move it, while the PR AUC of 0.186 compares false alarms to the people the model flags, which shows how often a flag is wrong. Random guessing would get a PR AUC of about 0.07, so the model beats that, but 0.186 is a lot less impressive than 0.714, which makes the PR curve the more honest picture when only 7% of people default.

In [9]:
principal = np.exp(X_te["log_loan"]).values
y_true = y_te.values
REP_COST = 250

thresholds = np.linspace(0, 1, 101)
costs = []
for t in thresholds:
    flag = lasso_prob >= t
    fn = (~flag) & (y_true == 1)
    fp = flag & (y_true == 0)
    costs.append((0.6 * principal[fn]).sum() + (0.08 * principal[fp]).sum() + REP_COST * fp.sum())
costs = np.array(costs)

best_t = thresholds[costs.argmin()]
print(f"Best threshold: {best_t:.2f}")
print(f"Cost at best threshold: {costs.min():,.0f}")
print(f"Cost at 0.5: {costs[50]:,.0f}")
print(metrics.confusion_matrix(y_te, (lasso_prob >= best_t).astype(int)))

Best threshold: 0.14
Cost at best threshold: 1,102,710
Cost at 0.5: 1,328,838
[[1529  144]
 [  87   40]]


I set the reputational cost at 250 dollars for each good applicant who gets turned down, since that person might complain, leave a bad review or take future business somewhere else, and that harm doesn't depend on how big their loan was. The cheapest threshold is 0.14, which is 0.36 below the usual 0.5, and it lowers the total cost from 1,328,838 dollars to 1,102,710 dollars, saving 226,128 dollars on the test set. At 0.14 the model catches 40 of the 127 defaults instead of just 1, but it also turns down 144 good applicants instead of none, which is worth it because a missed default costs way more than a turned-down good customer.

In [10]:
for loss in [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]:
    c = []
    for t in thresholds:
        flag = lasso_prob >= t
        fn = (~flag) & (y_true == 1)
        fp = flag & (y_true == 0)
        c.append((loss * principal[fn]).sum() + (0.08 * principal[fp]).sum() + REP_COST * fp.sum())
    print(f"Loss on default: {loss:.0%}, cost ratio: {loss / 0.08:.1f}, best threshold: {thresholds[np.argmin(c)]:.2f}")

Loss on default: 10%, cost ratio: 1.2, best threshold: 0.47
Loss on default: 20%, cost ratio: 2.5, best threshold: 0.28
Loss on default: 40%, cost ratio: 5.0, best threshold: 0.18
Loss on default: 60%, cost ratio: 7.5, best threshold: 0.14
Loss on default: 80%, cost ratio: 10.0, best threshold: 0.10
Loss on default: 100%, cost ratio: 12.5, best threshold: 0.10


As a missed default gets more expensive compared to turning down a good applicant, the best threshold drops, going from 0.47 when the ratio is 1.2 down to 0.10 when the ratio is 10 or higher. The rule for the head of risk is to turn down any applicant whose chance of default is above 1 divided by 1 plus the cost ratio, so at a ratio of 7.5 that works out to about 0.12, which is close to the 0.14 found here, with the small gap coming from the reputational cost. The 0.5 default only makes sense if both mistakes cost the same, and since a default costs this lender much more, using 0.5 means approving too many risky loans.

(AI RESPONSE)

The right tool here is McNemar's test, not a comparison of the two accuracy numbers. Both models were scored on the same rows, so their results are paired. A two-proportion z-test would treat them as independent samples and give you the wrong p-value. McNemar ignores the rows where both models agree and asks whether the rows where they disagree split unevenly in the forest's favor.

Swap y_pred_rf and y_pred_lr for whatever you named your prediction arrays.

```python
import numpy as np  # array math and element-wise comparisons
import pandas as pd  # builds a readable 2x2 table of paired outcomes
from scipy import stats  # binomial and chi-square distributions for the p-values

y_true = np.asarray(y_test)  # turn the test labels into a plain numpy array
rf_correct = np.asarray(y_pred_rf) == y_true  # True on each row the random forest classified correctly
lr_correct = np.asarray(y_pred_lr) == y_true  # True on each row the logistic model classified correctly

table = pd.crosstab(rf_correct, lr_correct, rownames=["RF correct"], colnames=["LR correct"])  # cross-tabulate the two models' right/wrong outcomes row by row
print(table)  # show how often the models agree and disagree

b = int(np.sum(rf_correct & ~lr_correct))  # rows the forest got right and the logistic model got wrong
c = int(np.sum(~rf_correct & lr_correct))  # rows the logistic model got right and the forest got wrong
n_disc = b + c  # total disagreements, which are the only rows that carry evidence about which model is better

print(f"RF accuracy: {rf_correct.mean():.4f}")  # forest accuracy, as a sanity check against your 0.9?? figure
print(f"LR accuracy: {lr_correct.mean():.4f}")  # logistic accuracy, same check
print(f"Discordant rows: RF-only correct = {b}, LR-only correct = {c}")  # the two counts the test is built on

if n_disc == 0:  # if the models never disagree there is nothing to test
    print("Models never disagree, so no test is possible: they are identical on this test set.")  # report it and stop here
else:  # otherwise run the test
    p_two_sided = stats.binomtest(b, n_disc, 0.5, alternative="two-sided").pvalue  # exact McNemar: is the split of disagreements different from 50/50 in either direction?
    p_rf_better = stats.binomtest(b, n_disc, 0.5, alternative="greater").pvalue  # exact McNemar, one-sided: does the split favor the forest?
    chi2_stat = (abs(b - c) - 1) ** 2 / n_disc  # classic McNemar chi-square statistic with continuity correction
    p_chi2 = stats.chi2.sf(chi2_stat, df=1)  # upper-tail p-value from a chi-square with 1 degree of freedom
    print(f"Exact McNemar p (two-sided): {p_two_sided:.4g}")  # the number to report by default
    print(f"Exact McNemar p (one-sided, RF better): {p_rf_better:.4g}")  # only report this one if you committed to that direction before looking
    print(f"Chi-square McNemar p (continuity-corrected): {p_chi2:.4g}")  # large-sample approximation, should roughly match the exact two-sided value
```

**What the test assumes about the two sets of predictions**

- Paired predictions. Both arrays are predictions on the exact same rows in the same order, with no shuffling and no dropped rows in one of them.
- Independent rows. Each loan is a separate draw. If the same borrower appears more than once, or the test set is a time slice where defaults cluster, the p-value will be too optimistic.
- Only disagreements matter. Under the null hypothesis, a disagreement is equally likely to favor either model, so the forest-only count b follows Binomial(b + c, 0.5). That is why the exact version uses binomtest. The chi-square version is fine when b + c is roughly 25 or more.
- Fixed models. The test treats both fitted models as fixed and asks only whether this test set's difference is noise. It says nothing about whether the forest would win again if both were retrained on a different sample.
- A clean test set. It assumes the test set was not used for tuning or choosing either model. If it was, the p-value is biased.

A risk-analyst note before this goes upstairs. In credit, a 0.9-something accuracy can mostly reflect the share of loans that don't default. A model that approves everyone may score about the same. A significant McNemar result only means one model makes fewer threshold errors. It does not show that the model ranks risk better, is better calibrated, or catches more of the defaults that cost money. I'd put AUC/Gini, the default-class recall, and a calibration check next to this result.

**The sentence for the head of risk**

"On the same held-out loans, the random forest was right on [b] cases the logistic model missed and wrong on [c] cases it caught, and McNemar's exact test gives p = [p_two_sided], so we [do / do not] have evidence that the forest's accuracy edge is real rather than test-set noise, though accuracy alone doesn't tell us which model better separates good loans from defaults."

In [11]:
from scipy import stats

rf_correct = rf_pred == y_true
lasso_correct = lasso_pred == y_true

b = int(np.sum(rf_correct & ~lasso_correct))
c = int(np.sum(~rf_correct & lasso_correct))

print(f"b, forest right and LASSO wrong: {b}")
print(f"c, forest wrong and LASSO right: {c}")
print(f"b + c: {b + c}")
if b + c > 0:
    print(f"Exact McNemar p-value: {stats.binomtest(b, b + c, 0.5).pvalue:.4f}")

b, forest right and LASSO wrong: 0
c, forest wrong and LASSO right: 1
b + c: 1
Exact McNemar p-value: 1.0000


The AI picked McNemar's test, which was right since both models predicted the same rows, and here b is 0, c is 1 and b + c is 1, so the exact binomial test was used because there was only one disagreement, and it gave a p-value of 1.0000. That means the Step 2.2 verdict holds, since the forest is not better than LASSO and the lender can keep the cheaper model that is easier to explain. The catch is that this test only counts mistakes and not what they cost, so a better check would compare total cost or PR AUC for both models on the same resampled test rows.

## AI Transcript

**Model:** Claude Opus 5.5
**Date:** November 24, 2026

### Prompt as sent

[Prep] Act as a model risk analyst at a bank, reviewing a credit scoring model before it goes into production.

[Request] I have a fitted classifier producing default probabilities on a test set with about 7% prevalence. Write Python that assesses CALIBRATION -- a reliability diagram binning predicted probability against observed frequency, plus the Brier score and its decomposition -- and then applies both Platt scaling and isotonic regression, reporting whether either improves calibration.

[Iterate] scikit-learn, numpy, matplotlib. Reuse my y_test and y_prob. Handle the sparse high-probability bins that low prevalence creates rather than letting them produce a misleading plot.

[Mechanism Check] Explain why a model can rank applicants correctly -- high AUC -- and still be badly calibrated, and why that distinction decides whether my cost-optimal threshold is trustworthy.

[Evaluate] Given my reliability diagram, is my Phase 3 threshold valid? If not, what should I do before using it?

### Model's reply

I don't have the actual reliability diagram or the Phase 3 threshold yet, so the code and explanation come first and the Evaluate section gives the verdict logic. Once you run this, paste the printed table and the threshold value, and I'll give a concrete answer. I've labeled the code by cell since I don't know the box numbers in the assignment. Let me know the real labels, and also flag any function here that wasn't used in the lab, so I can swap it for one that was.

**Code**

Cell 1: setup

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss, roc_auc_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold

y_test = np.asarray(y_test).astype(int)
y_prob = np.clip(np.asarray(y_prob, dtype=float), 1e-6, 1 - 1e-6)
print("n =", len(y_test), "| prevalence =", round(y_test.mean(), 4), "| mean predicted =", round(y_prob.mean(), 4))
```

The last line is already a calibration check. If the mean predicted PD is far from about 0.07, the model is miscalibrated on average before you look at any bins.

Cell 2: binning with sparse-bin handling

```python
def reliability_table(y, p, n_bins=10, strategy="quantile", min_count=30):
    if strategy == "quantile":
        edges = np.unique(np.quantile(p, np.linspace(0, 1, n_bins + 1)))
    else:
        edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1], right=True), 0, len(edges) - 2)
    rows = []
    z = 1.96
    for b in range(len(edges) - 1):
        m = idx == b
        n = m.sum()
        if n == 0:
            continue
        obs = y[m].mean()
        denom = 1 + z**2 / n
        center = (obs + z**2 / (2 * n)) / denom
        half = z * np.sqrt(obs * (1 - obs) / n + z**2 / (4 * n**2)) / denom
        rows.append([n, p[m].mean(), obs, center - half, center + half, n >= min_count])
    return np.array(rows)
```

This handles the sparse bins in three ways. Quantile bins give every bin the same number of applicants, so the high-PD region never ends up with a bin of 4 loans. Wilson intervals show how uncertain each observed default rate is, and they behave well when the rate is near 0. Bins with fewer than min_count loans get flagged, so the plot can show them as unreliable instead of drawing them like real evidence.

Cell 3: Brier decomposition

```python
def brier_decomp(y, p, n_bins=10):
    t = reliability_table(y, p, n_bins, "quantile", 0)
    n, pbar, obar = t[:, 0], t[:, 1], t[:, 2]
    N = n.sum()
    base = y.mean()
    bs = brier_score_loss(y, p)
    rel = np.sum(n * (pbar - obar) ** 2) / N
    res = np.sum(n * (obar - base) ** 2) / N
    unc = base * (1 - base)
    return {"brier": bs, "reliability": rel, "resolution": res, "uncertainty": unc, "residual": bs - (rel - res + unc)}
```

This is the Murphy decomposition: Brier = reliability − resolution + uncertainty.

- Reliability is the calibration error, and lower is better.
- Resolution measures how well the model separates risk groups, and higher is better.
- Uncertainty is 0.07 × 0.93 ≈ 0.065 for any model, because it depends only on prevalence.

The residual is nonzero because binning ignores variation within each bin. If the residual is large next to the reliability term, use more bins.

Cell 4: reliability diagram

```python
def plot_reliability(y, p, label, ax_q, ax_u, min_count=30):
    tq = reliability_table(y, p, 10, "quantile", min_count)
    ax_q.errorbar(tq[:, 1], tq[:, 2], yerr=[tq[:, 2] - tq[:, 3], tq[:, 4] - tq[:, 2]], fmt="o-", capsize=3, label=label)
    tu = reliability_table(y, p, 10, "uniform", min_count)
    ok = tu[:, 5].astype(bool)
    ax_u.errorbar(tu[ok, 1], tu[ok, 2], yerr=[tu[ok, 2] - tu[ok, 3], tu[ok, 4] - tu[ok, 2]], fmt="o-", capsize=3, label=label)
    ax_u.scatter(tu[~ok, 1], tu[~ok, 2], facecolors="none", edgecolors="gray")
    for row in tu:
        ax_u.annotate(int(row[0]), (row[1], row[2]), fontsize=7, xytext=(3, 3), textcoords="offset points")

fig, (ax_q, ax_u) = plt.subplots(1, 2, figsize=(13, 5.5))
plot_reliability(y_test, y_prob, "Uncalibrated", ax_q, ax_u)
hi = max(np.quantile(y_prob, 0.99), 0.15)
ax_q.plot([0, hi], [0, hi], "k--", lw=1)
ax_q.set_xlim(0, hi); ax_q.set_ylim(0, hi)
ax_q.set_title("Quantile bins (equal count) — primary view")
ax_u.plot([0, 1], [0, 1], "k--", lw=1)
ax_u.set_title("Uniform bins — hollow = n < 30, unreliable")
for a in (ax_q, ax_u):
    a.set_xlabel("Mean predicted PD"); a.set_ylabel("Observed default rate"); a.legend()
plt.tight_layout(); plt.show()
```

The left panel is the one to read. It zooms to the range where 99% of predictions fall, because with 7% prevalence most of the action is between 0 and 0.2. The right panel keeps the standard uniform-bin view but labels each bin's count and shows bins with fewer than 30 loans as hollow circles. A wild point at a predicted PD of 0.85 based on 6 loans is then visibly noise rather than a finding.

Cell 5: Platt scaling and isotonic regression, out-of-fold

```python
def logit(p):
    return np.log(p / (1 - p))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
p_platt = np.zeros_like(y_prob)
p_iso = np.zeros_like(y_prob)
for fit_idx, eval_idx in skf.split(y_prob.reshape(-1, 1), y_test):
    lr = LogisticRegression(C=1e6)
    lr.fit(logit(y_prob[fit_idx]).reshape(-1, 1), y_test[fit_idx])
    p_platt[eval_idx] = lr.predict_proba(logit(y_prob[eval_idx]).reshape(-1, 1))[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
    iso.fit(y_prob[fit_idx], y_test[fit_idx])
    p_iso[eval_idx] = iso.predict(y_prob[eval_idx])
p_iso = np.clip(p_iso, 1e-6, 1 - 1e-6)
```

This cell matters most from a model-risk standpoint. If you fit the calibrator on y_test and then score it on y_test, isotonic regression in particular will look nearly perfect because it memorizes the data. Each calibrator here is fit on 4/5 of the test set and scored on the held-out 1/5, so the comparison is honest.

Cell 6: comparison and bootstrap significance

```python
models = {"Uncalibrated": y_prob, "Platt": p_platt, "Isotonic": p_iso}
print(f"{'Model':<14}{'Brier':>9}{'Reliab':>10}{'Resol':>10}{'LogLoss':>10}{'AUC':>8}")
for name, p in models.items():
    d = brier_decomp(y_test, p)
    print(f"{name:<14}{d['brier']:>9.5f}{d['reliability']:>10.6f}{d['resolution']:>10.6f}{log_loss(y_test, p):>10.4f}{roc_auc_score(y_test, p):>8.4f}")

rng = np.random.default_rng(0)
for name in ["Platt", "Isotonic"]:
    diffs = []
    for _ in range(2000):
        b = rng.integers(0, len(y_test), len(y_test))
        diffs.append(brier_score_loss(y_test[b], y_prob[b]) - brier_score_loss(y_test[b], models[name][b]))
    lo, hi_ci = np.percentile(diffs, [2.5, 97.5])
    verdict = "improves" if lo > 0 else ("worsens" if hi_ci < 0 else "no significant change")
    print(f"{name}: Brier reduction 95% CI [{lo:.5f}, {hi_ci:.5f}] -> {verdict}")

fig, (ax_q, ax_u) = plt.subplots(1, 2, figsize=(13, 5.5))
for name, p in models.items():
    plot_reliability(y_test, p, name, ax_q, ax_u)
ax_q.plot([0, hi], [0, hi], "k--", lw=1); ax_q.set_xlim(0, hi); ax_q.set_ylim(0, hi)
ax_u.plot([0, 1], [0, 1], "k--", lw=1)
for a in (ax_q, ax_u):
    a.set_xlabel("Mean predicted PD"); a.set_ylabel("Observed default rate"); a.legend()
plt.tight_layout(); plt.show()
```

How to read the table:

- AUC should barely move. Platt scaling is monotone, so AUC stays the same. Isotonic regression creates ties, so its AUC can drop slightly.
- Watch the reliability column. "Improves" should mean reliability drops and the bootstrap interval for the Brier reduction sits entirely above zero.
- Prefer Platt if the two tie. Isotonic regression is more flexible but overfits when there are few defaults. At 7% prevalence, a test set of 3,000 has only about 210 defaults to fit a step function to.

**Mechanism Check**

AUC measures one thing: if you pick a random defaulter and a random non-defaulter, how often does the defaulter get the higher score? Only the order of the scores matters. You could square every probability, halve them, or push them all toward 0.5, and AUC would not change, because the ranking would stay the same. Calibration asks a different question: of the applicants scored at 10%, do about 10% actually default? That depends on

In [12]:
from sklearn.calibration import calibration_curve

obs, pred = calibration_curve(y_te, lasso_prob, n_bins=10, strategy="quantile")
print(pd.DataFrame({"Mean predicted": pred, "Observed default rate": obs}).round(3))
print(f"Average predicted: {lasso_prob.mean():.3f}")
print(f"Actual default rate: {y_te.mean():.3f}")
print(f"Brier score: {metrics.brier_score_loss(y_te, lasso_prob):.4f}")

   Mean predicted  Observed default rate
0           0.021                  0.017
1           0.031                  0.033
2           0.039                  0.044
3           0.047                  0.028
4           0.056                  0.039
5           0.065                  0.039
6           0.076                  0.094
7           0.091                  0.072
8           0.120                  0.128
9           0.194                  0.211
Average predicted: 0.074
Actual default rate: 0.071
Brier score: 0.0622


The LASSO model is well calibrated, since its average predicted default chance of 0.074 is close to the real default rate of 0.071, and in each bin the predicted and observed rates stay close together. Near the Phase 3 threshold of 0.14, the bins at 0.120 and 0.194 predicted had real default rates of 0.128 and 0.211, so the probabilities mean what they say around the cutoff. That means the 0.14 threshold passes the calibration check and does not need to be recomputed, and the Brier score of 0.0622 is a little better than the 0.066 a model would get by guessing the average rate for everyone.

## Task 4.2: The recommendation

**To the head of risk:**

The vendor's deal should not be signed just because of the 93% accuracy number, since a model that approves everyone gets 92.9% on this data and still catches zero defaults. When only 7% of loans default, accuracy can't tell a good model from a useless one, so the vendor should send its predicted probabilities on a test set and be judged on how much money it saves.

A simple LASSO model shows what a real model is worth compared to approving everyone. Its accuracy is the same as the null model, but its AUC of 0.714 and PR AUC of 0.186, compared to about 0.07 for random guessing, show it does pick up real risk. A random forest did no better and only disagreed with LASSO on one test loan, so the simpler model makes more sense and is easier to explain to a regulator.

The best threshold is 0.14 instead of the usual 0.5. That number assumes a default loses 60% of the loan and turning down a good applicant loses 8% of the loan in interest plus 250 dollars in reputational cost. With those costs, the 0.14 cutoff cuts test set losses from 1,328,838 dollars to 1,102,710 dollars, and the model's probabilities line up well with real default rates near that cutoff, so the threshold holds up.

The one thing that would change this answer is the real loss on a default, because if it's much lower than 60%, the best threshold moves back toward 0.5 and most of the savings go away.